# Phase 2: Data Preprocessing & Feature Engineering

This notebook handles the cleaning and transformation of the Metro Interstate Traffic Volume dataset. We will extract time-based features and encode weather categories to prepare the data for machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import LabelEncoder
import joblib

# Set visual style
sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
df = pd.read_csv('../dataset/Metro_Interstate_Traffic_Volume.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Basic Cleaning
Check for missing values and duplicates.

In [ ]:
print("Missing values:\n", df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

df.drop_duplicates(inplace=True)
print(f"Shape after removing duplicates: {df.shape}")

## 3. Feature Engineering
Converting `date_time` into numerical features that the model can interpret.

In [ ]:
df['date_time'] = pd.to_datetime(df['date_time'])

# Extract time features
df['hour'] = df['date_time'].dt.hour
df['day'] = df['date_time'].dt.day
df['month'] = df['date_time'].dt.month
df['weekday'] = df['date_time'].dt.weekday
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)

# Define rush hours (typical 7-9 AM and 4-6 PM)
df['is_rush_hour'] = df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 18) else 0)

df.head()

## 4. Encoding Categorical Data
The `weather_main` column is text. We need to convert it to numbers using Label Encoding.

In [ ]:
le_weather = LabelEncoder()
df['weather_main_enc'] = le_weather.fit_transform(df['weather_main'])

# Save the encoder to use later in the API
os.makedirs('../models', exist_ok=True)
joblib.dump(le_weather, '../models/weather_encoder.joblib')

print("Weather categories:", le_weather.classes_)

## 5. Visualizing the Patterns
Let's see the average traffic volume by hour to confirm our 'rush hour' assumption.

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='hour', y='traffic_volume', color='royalblue')
plt.title('Average Traffic Volume by Hour')
plt.show()

## 6. Saving Processed Data
We will save this version to be used by the training script.

In [ ]:
os.makedirs('../dataset/processed', exist_ok=True)

# Select only the features we'll use for training
features = ['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main_enc', 'hour', 'day', 'month', 'weekday', 'is_weekend', 'is_rush_hour', 'traffic_volume']

df[features].to_csv('../dataset/processed/cleaned_traffic_data.csv', index=False)
print("Success! Cleaned data saved to dataset/processed/cleaned_traffic_data.csv")